# 05 — Final Load Prep: KPI Computation & Tableau Export

This notebook computes all 7 KPIs defined in the project framework, creates a Tableau-ready dataset with clean column names and pre-computed metrics, and exports it for dashboard building.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_olist_dataset.csv"

df = pd.read_csv(DATA_PATH, parse_dates=[
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
])
print(f"Loaded: {len(df):,} rows × {df.columns.size} columns")

Loaded: 110,197 rows × 48 columns


---
## 1. Order-Level KPI Table

One row per order with all KPIs pre-computed. This is the primary file for Tableau.

In [2]:
order_kpi = df.groupby("order_id").agg(
    customer_unique_id=("customer_unique_id", "first"),
    customer_state=("customer_state", "first"),
    customer_city=("customer_city", "first"),
    purchase_date=("order_purchase_timestamp", "first"),
    purchase_month=("purchase_month", "first"),
    purchase_year=("purchase_year", "first"),
    purchase_month_num=("purchase_month_num", "first"),
    purchase_weekday=("purchase_weekday", "first"),
    purchase_hour=("purchase_hour", "first"),
    order_item_count=("order_item_id", "max"),
    total_items_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    total_order_value=("total_item_value", "sum"),
    total_payment=("total_payment_value", "first"),
    primary_payment_type=("primary_payment_type", "first"),
    max_installments=("max_installments", "first"),
    product_category_main=("product_category", lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else "mixed"),
    review_score=("review_score", "first"),
    actual_delivery_days=("actual_delivery_days", "first"),
    delivery_delay_days=("delivery_delay_days", "first"),
    is_on_time=("is_on_time", "first"),
    seller_state=("seller_state", "first"),
).reset_index()

print(f"Order KPI table: {len(order_kpi):,} rows × {order_kpi.columns.size} columns")

Order KPI table: 96,478 rows × 23 columns


## 2. KPI Computation — All 7 Metrics

In [3]:
# --- KPI 1: Monthly Revenue ---
monthly_revenue = order_kpi.groupby("purchase_month")["total_payment"].sum().reset_index()
monthly_revenue.columns = ["month", "revenue_brl"]
monthly_revenue = monthly_revenue[monthly_revenue["month"] >= "2017-01"]

# --- KPI 2: Average Order Value ---
aov = order_kpi["total_payment"].mean()
print(f"KPI 2 — Average Order Value:     BRL {aov:,.2f}")

# --- KPI 3: On-Time Delivery Rate ---
ot_rate = order_kpi["is_on_time"].mean()
print(f"KPI 3 — On-Time Delivery Rate:    {ot_rate:.1%}")

# --- KPI 4: Customer Satisfaction (CSAT) ---
csat = order_kpi["review_score"].mean()
print(f"KPI 4 — Avg Review Score (CSAT):  {csat:.2f} / 5")

# --- KPI 5: Repeat Purchase Rate ---
cust_orders = order_kpi.groupby("customer_unique_id").size()
repeat_rate = (cust_orders > 1).mean()
print(f"KPI 5 — Repeat Purchase Rate:     {repeat_rate:.1%}")

# --- KPI 6: Avg Delivery Delay ---
avg_delay = order_kpi["delivery_delay_days"].dropna().mean()
print(f"KPI 6 — Avg Delivery Delay:       {avg_delay:.1f} days (negative = early)")

# --- KPI 7: Freight-to-Price Ratio ---
avg_freight_ratio = order_kpi["total_freight"].sum() / order_kpi["total_items_price"].replace(0, np.nan).sum()
print(f"KPI 7 — Platform Freight Ratio:   {avg_freight_ratio:.1%}")

KPI 2 — Average Order Value:     BRL 159.86
KPI 3 — On-Time Delivery Rate:    93.2%
KPI 4 — Avg Review Score (CSAT):  4.16 / 5
KPI 5 — Repeat Purchase Rate:     3.0%
KPI 6 — Avg Delivery Delay:       -11.9 days (negative = early)
KPI 7 — Platform Freight Ratio:   16.6%


## 3. Segmented KPI Tables for Tableau

Pre-aggregated tables that make Tableau filter/parameter construction easier.

In [4]:
# --- State-level KPIs ---
state_kpi = order_kpi.groupby("customer_state").agg(
    total_revenue=("total_payment", "sum"),
    order_count=("order_id", "count"),
    avg_order_value=("total_payment", "mean"),
    avg_review_score=("review_score", "mean"),
    on_time_rate=("is_on_time", "mean"),
    avg_delivery_days=("actual_delivery_days", "mean"),
    avg_delivery_delay=("delivery_delay_days", "mean"),
).reset_index().round(2)

print(f"State KPI table: {len(state_kpi)} states\n")
state_kpi.sort_values("total_revenue", ascending=False).head(10)

State KPI table: 27 states



,customer_state,total_revenue,order_count,avg_order_value,avg_review_score,on_time_rate,avg_delivery_days,avg_delivery_delay
25,SP,5770266.19,40501,142.48,4.25,0.95,8.30,-11.08
18,RJ,2055690.45,12350,166.45,3.97,0.88,14.85,-11.76
10,MG,1819277.61,11354,160.23,4.19,0.95,11.54,-13.24
22,RS,861802.40,5345,161.24,4.18,0.94,14.82,-13.91
17,PR,781919.55,4923,158.83,4.24,0.96,11.53,-13.31
23,SC,595208.40,3546,167.85,4.13,0.92,14.48,-11.50
4,BA,591270.60,3256,181.59,3.93,0.88,18.87,-10.79
6,DF,346146.17,2080,166.42,4.13,0.94,12.51,-12.05
8,GO,334294.22,1957,170.82,4.11,0.93,15.15,-12.19
7,ES,317682.65,1995,159.24,4.08,0.89,15.33,-10.50


In [5]:
# --- Category-level KPIs ---
cat_kpi = order_kpi.groupby("product_category_main").agg(
    total_revenue=("total_payment", "sum"),
    order_count=("order_id", "count"),
    avg_order_value=("total_payment", "mean"),
    avg_review_score=("review_score", "mean"),
    on_time_rate=("is_on_time", "mean"),
).reset_index().round(2)

print(f"Category KPI table: {len(cat_kpi)} categories\n")
cat_kpi.sort_values("total_revenue", ascending=False).head(10)

Category KPI table: 72 categories



,product_category_main,total_revenue,order_count,avg_order_value,avg_review_score,on_time_rate
43,health_beauty,1413962.84,8621,164.03,4.23,0.92
71,watches_gifts,1257930.55,5455,230.60,4.13,0.93
7,bed_bath_table,1241203.08,9239,134.34,4.00,0.93
65,sports_leisure,1115692.71,7478,149.20,4.24,0.93
15,computers_accessories,1037148.01,6520,159.07,4.08,0.94
39,furniture_decor,884646.29,6208,142.50,4.07,0.93
49,housewares,754917.89,5670,133.14,4.21,0.95
22,cool_stuff,688757.24,3516,195.89,4.23,0.94
5,auto,671719.35,3804,176.58,4.15,0.93
42,garden_tools,566463.25,3411,166.07,4.19,0.93


In [6]:
# --- Monthly trend KPIs ---
monthly_kpi = order_kpi.groupby("purchase_month").agg(
    revenue=("total_payment", "sum"),
    orders=("order_id", "count"),
    aov=("total_payment", "mean"),
    avg_review=("review_score", "mean"),
    on_time_rate=("is_on_time", "mean"),
    avg_delivery_days=("actual_delivery_days", "mean"),
).reset_index().round(2)
monthly_kpi = monthly_kpi[monthly_kpi["purchase_month"] >= "2017-01"]

print(f"Monthly KPI table: {len(monthly_kpi)} months")
monthly_kpi.tail(6)

Monthly KPI table: 20 months


,purchase_month,revenue,orders,aov,avg_review,on_time_rate,avg_delivery_days
17,2018-03,1120678.00,7003,160.03,3.81,0.81,15.87
18,2018-04,1132933.95,6798,166.66,4.21,0.95,11.05
19,2018-05,1128836.69,6749,167.26,4.24,0.93,10.96
20,2018-06,1012090.68,6099,165.94,4.31,0.99,8.77
21,2018-07,1027903.86,6159,166.89,4.32,0.97,8.50
22,2018-08,985414.28,6351,155.16,4.31,0.94,7.29


In [7]:
# --- Payment type KPIs ---
pay_kpi = order_kpi.groupby("primary_payment_type").agg(
    total_revenue=("total_payment", "sum"),
    order_count=("order_id", "count"),
    avg_order_value=("total_payment", "mean"),
    avg_review=("review_score", "mean"),
    avg_installments=("max_installments", "mean"),
).reset_index().round(2)

pay_kpi

,primary_payment_type,total_revenue,order_count,avg_order_value,avg_review,avg_installments
0,boleto,2769838.18,19190,144.34,4.16,1.0
1,credit_card,12280373.97,74275,165.34,4.16,3.5
2,debit_card,201721.24,1436,140.47,4.25,1.0
3,voucher,157495.41,1498,105.14,4.11,1.0


---
## 4. Export All Tableau-Ready Files

In [8]:
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

exports = {
    "tableau_orders_kpi.csv": order_kpi,
    "tableau_state_kpi.csv": state_kpi,
    "tableau_category_kpi.csv": cat_kpi,
    "tableau_monthly_kpi.csv": monthly_kpi,
    "tableau_payment_kpi.csv": pay_kpi,
}

for filename, data in exports.items():
    path = output_dir / filename
    data.to_csv(path, index=False)
    print(f"  ✓ {filename:35s} {len(data):>8,} rows  ({path.stat().st_size / 1024:.0f} KB)")

print(f"\nAll Tableau-ready files exported to {output_dir}")

  ✓ tableau_orders_kpi.csv                96,478 rows  (19526 KB)
  ✓ tableau_state_kpi.csv                     27 rows  (1 KB)
  ✓ tableau_category_kpi.csv                  72 rows  (3 KB)
  ✓ tableau_monthly_kpi.csv                   20 rows  (1 KB)
  ✓ tableau_payment_kpi.csv                    4 rows  (0 KB)

All Tableau-ready files exported to /Users/hemanth10etii/Coding/DVACap/data/processed


## 5. KPI Summary Card

Quick reference for the dashboard executive view.

In [9]:
print("=" * 55)
print("  OLIST MARKETPLACE — KPI SCORECARD")
print("=" * 55)
print(f"  Total Revenue:          BRL {order_kpi['total_payment'].sum():>14,.2f}")
print(f"  Total Orders:           {len(order_kpi):>14,}")
print(f"  Unique Customers:       {order_kpi['customer_unique_id'].nunique():>14,}")
print(f"  Unique Categories:      {order_kpi['product_category_main'].nunique():>14,}")
print(f"  Date Range:             2017-01 → 2018-08")
print("-" * 55)
print(f"  Avg Order Value:        BRL {aov:>14,.2f}")
print(f"  CSAT (Avg Review):      {csat:>14.2f} / 5")
print(f"  On-Time Delivery:       {ot_rate:>13.1%}")
print(f"  Avg Delivery Days:      {order_kpi['actual_delivery_days'].mean():>14.1f}")
print(f"  Avg Delivery Delay:     {avg_delay:>14.1f} days")
print(f"  Repeat Purchase Rate:   {repeat_rate:>13.1%}")
print(f"  Platform Freight Ratio: {avg_freight_ratio:>13.1%}")
print("=" * 55)

  OLIST MARKETPLACE — KPI SCORECARD
  Total Revenue:          BRL  15,422,461.77
  Total Orders:                   96,478
  Unique Customers:               93,358
  Unique Categories:                  72
  Date Range:             2017-01 → 2018-08
-------------------------------------------------------
  Avg Order Value:        BRL         159.86
  CSAT (Avg Review):                4.16 / 5
  On-Time Delivery:               93.2%
  Avg Delivery Days:                12.1
  Avg Delivery Delay:              -11.9 days
  Repeat Purchase Rate:            3.0%
  Platform Freight Ratio:         16.6%
